In [ ]:
!pip install cupy-cuda11x --quiet

In [16]:
import heapq
import cupy as cp

# Sample text
text = "this is an example for huffman encoding"

# Step 1: Build frequency dictionary
frequency = {}
for char in text:
    frequency[char] = frequency.get(char, 0) + 1

# Step 2: Build Huffman Tree
heap = [[weight, [symbol, ""]] for symbol, weight in frequency.items()]
heapq.heapify(heap)

while len(heap) > 1:
    lo = heapq.heappop(heap)
    hi = heapq.heappop(heap)
    for pair in lo[1:]:
        pair[1] = '0' + pair[1]
    for pair in hi[1:]:
        pair[1] = '1' + pair[1]
    heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])

# Step 3: Extract Huffman codes
huffman_codes = dict()
for symbol, code in heap[0][1:]:
    huffman_codes[symbol] = code

print("Huffman Codes:")
print(huffman_codes)

# Step 4: Encode text using Huffman codes
encoded_text = ''.join([huffman_codes[char] for char in text])
print("\nEncoded Bitstream:")
print(encoded_text)

# Step 5: Use GPU for parallelized encoding (CuPy)
# Prepare char->code map for GPU
char_list = list(huffman_codes.keys())
code_list = list(huffman_codes.values())

# Create GPU-compatible arrays
char_arr = cp.array([ord(c) for c in char_list], dtype=cp.uint8)
text_arr = cp.array([ord(c) for c in text], dtype=cp.uint8)

# Map text to encoded bitstream using GPU
encoded_gpu = cp.array([huffman_codes[chr(c)] for c in text_arr])

print("\nGPU Encoded Bitstream (First 10):")
print(encoded_gpu[:10])  # Show first 10 encoded bits


/usr/local/lib/python3.11/dist-packages/cupy/_environment.py:541: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


Huffman Codes:
{'g': '00000', 'l': '00001', 'h': '0001', 'm': '0010', 'o': '0011', 'n': '010', 'p': '01100', 'r': '01101', 's': '0111', 't': '10000', 'u': '10001', 'a': '1001', ' ': '101', 'e': '1100', 'f': '1101', 'i': '1110', 'x': '11110', 'c': '111110', 'd': '111111'}

Encoded Bitstream:
1000000011110011110111100111101100101010111001111010010010011000000111001011101001101101101000110001110111010010100101010111000101111100011111111111001000000


CUDARuntimeError: cudaErrorInsufficientDriver: CUDA driver version is insufficient for CUDA runtime version